# 🌍 Interactive Climate Change Dashboard: Data Analysis & Modeling
### Exploratory Data Analysis, Trend Forecasting & Anthropogenic Attribution
**Author:** Dev Solanki  
**Datasets:** Berkeley Earth Surface Temperature, NOAA GML Mauna Loa Keeling Curve, IPCC AR6 Scenarios
**Repository:** [Interactive-climate-change-dashboard](https://github.com/divyasolankigla-pixel/Interactive-climate-change-dashboard)

## 1. Project Overview & Problem Statement
This research and exploratory data analysis notebook investigates multi-century global climate indicators (1750–2024). We quantify the rate of surface temperature rise, greenhouse gas emissions, sea level changes, and the statistical coupling between atmospheric carbon dioxide ($CO_2$) concentration and global thermal anomalies.

In [1]:
import os, json, math
import csv

data_dir = os.path.join('static', 'data')
print('Available JSON data assets:', os.listdir(data_dir))


## 2. Global Temperature Anomaly & Acceleration Analysis
We compute global land and ocean temperature anomalies relative to the pre-industrial baseline (1850–1900).

In [2]:
with open(os.path.join(data_dir, 'global_climate.json'), 'r') as f:
    climate_data = json.load(f)

series = climate_data['yearly_series']
print(f"Total observational years: {len(series)} ({series[0]['year']} to {series[-1]['year']})")
recent = series[-1]
print(f"Latest Land Anomaly ({recent['year']}): +{recent['anomaly_preind_land']}°C")
print(f"Pre-industrial Baseline (1850-1900): {climate_data['baselines']['preindustrial_land_1850_1900']}°C")


## 3. Pearson Correlation: Atmospheric $CO_2$ vs Temperature Anomaly
We measure the Pearson correlation coefficient ($r$) and coefficient of determination ($R^2$) between Mauna Loa atmospheric $CO_2$ (ppm) and surface temperature anomaly (°C).

In [3]:
with open(os.path.join(data_dir, 'eda_metrics.json'), 'r') as f:
    eda = json.load(f)

corr_res = eda['correlation_analysis']
for pair in corr_res['pairs']:
    print(f"{pair['var1']} vs {pair['var2']}:")
    print(f"   Pearson r = {pair['correlation_r']}, R² = {pair['r_squared']}")
    print(f"   Interpretation: {pair['interpretation']}\n")


## 4. Warming Acceleration Rates Across Historical Epochs
Comparing the rate of temperature rise (°C per decade) across consecutive periods demonstrates that modern warming has accelerated dramatically since 1980.

In [4]:
for rate in eda['warming_rates']:
    print(f"{rate['period']} ({rate['years_span']}) -> +{rate['deg_c_per_decade']}°C/decade ({rate['acceleration_factor_vs_1850']}x speed)")


## 5. IPCC AR6 Scenario Simulator & Carbon Budget Formulation
We formulate the Transient Climate Response to Cumulative Carbon Emissions (TCRE):
$$\Delta T = T_{current} + \left(\frac{\text{Cumulative } CO_2}{1000}\right) \times 0.45$$
Using this model, we project 2100 temperatures under user-defined decarbonization pathways.

In [5]:
def simulate_pathway(peak_year, decay_rate):
    emiss = 39.0
    cum = 0.0
    for y in range(2024, 2101):
        emiss = emiss * 1.008 if y <= peak_year else max(0.0, emiss * (1.0 - decay_rate/100.0))
        cum += emiss
    warming = 1.28 + (cum / 1000.0) * 0.45
    return round(warming, 2), round(cum, 1)

paris_w, paris_cum = simulate_pathway(2025, 7.5)
current_w, current_cum = simulate_pathway(2035, 2.0)
print(f"Paris 1.5°C Pathway (Peak 2025, -7.5%/yr): 2100 Warming = +{paris_w}°C (Cum: {paris_cum} Gt)")
print(f"Current Policies Pathway (Peak 2035, -2.0%/yr): 2100 Warming = +{current_w}°C (Cum: {current_cum} Gt)")
